In [26]:
!pip install -q torch torchvision scikit-learn bloqade

In [27]:
import numpy as np
# import juliacall
# bloqade_jl = juliacall.Main.seval('Bloqade')
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
import bloqade
from bloqade.atom_arrangement import Chain
from bloqade.emulate.ir.space import Space
from bloqade.ir.routine.bloqade import BloqadeEmulation
from bloqade.factory import rydberg_h
# from bloqade import evolve

In [28]:
from bloqade import (
    waveform,
    rydberg_h,
    piecewise_linear,
    piecewise_constant,
    constant,
    linear,
    var,
    cast,
    start,
    get_capabilities,
)
from bloqade.atom_arrangement import Chain
from bloqade.ir import (
    AnalogCircuit,
    Sequence,
    rydberg,
    Pulse,
    rabi,
    detuning,
    Field,
    Uniform,
)
from bloqade.ir.routine.base import Routine
from bloqade.ir.routine.params import Params, ScalarArg

In [29]:

# Define constants
dim_pca = 10
Δ_max = 6.0
num_examples = 1000
num_test_examples = 100

In [30]:
# Load MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=True, transform=transform)
test_dataset = datasets.MNIST('~/.pytorch/MNIST_data/', download=True, train=False, transform=transform)

In [31]:
# Create data loaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=100, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False)

In [32]:
# Perform PCA on training data
pca = PCA(n_components=dim_pca)
x_train_pca = pca.fit_transform(train_dataset.data.numpy().reshape(-1, 28*28))
x_test_pca = pca.transform(test_dataset.data.numpy().reshape(-1, 28*28))

In [33]:
# Scale PCA values to feasible range of local detuning
x_train_pca = x_train_pca / np.max(np.abs(x_train_pca)) * Δ_max
x_test_pca = x_test_pca / np.max(np.abs(x_test_pca)) * Δ_max

In [34]:
x_test_pca[:, 1:num_examples]

array([[ 1.8808865 , -0.1077646 , -0.78267452, ...,  1.18377565,
        -1.00442986, -0.46694598],
       [-2.40351597, -0.38411444,  1.00276539, ...,  0.58176512,
         0.37042237, -0.21911442],
       [-1.08367013,  0.16644886,  0.654219  , ..., -0.27663898,
         0.24836203, -0.34152847],
       ...,
       [ 1.50126618,  0.89318496, -1.04370864, ..., -0.88664707,
        -0.06186089, -2.01926141],
       [-0.27316307,  1.61689051, -0.63723799, ...,  0.41763249,
         0.67935121, -0.54133885],
       [-0.22766541,  1.77605237,  2.12734322, ..., -1.06852968,
         0.4303646 ,  0.13605877]])

In [35]:
# One-hot encode labels
encoder = OneHotEncoder(sparse_output=False)
y_train = encoder.fit_transform(train_dataset.targets.numpy().reshape(-1, 1))
y_test = encoder.transform(test_dataset.targets.numpy().reshape(-1, 1))

In [36]:
y_test[:, 1:num_examples]

array([[0., 0., 0., ..., 1., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [1., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [ ]:
# Define quantum reservoir computing (QRC) layer
class DetuningLayer(nn.Module):
    def __init__(self, atoms, readouts, Ω, t_start, t_end, step):
        super(DetuningLayer, self).__init__()
        self.atoms = atoms
        self.readouts = readouts
        self.Ω = Ω
        self.t_start = t_start
        self.t_end = t_end
        self.step = step
    def apply_layer(self, x):
      run_time = var("run_time")

      @waveform(run_time + 0.2)
      def delta(t, amp, omega):
        return np.sin(omega * t) * amp
      # example ham
      delta = delta.sample(0.05, "linear")
      ampl = piecewise_linear([0.1, run_time, 0.1], [0, 10, 10, 0])
      phase = piecewise_constant([2, 2], [0, np.pi])
      register = start.add_position((0, 0))
      atoms_position = (0, 0)
      # note x is with predict no just x
      prog = rydberg_h(
          self.atoms, detuning=delta, amplitude=x, phase=self.Ω, batch_params={}
      )
    # Simulate quantum dynamics and compute readouts
    # have to use bloqade quantum
    # This part is not implemented in Python, as it requires a quantum simulator
    # calculating steps
      # the hamiltonian would be calculate through the rydberg_h function given in bloqade python
      # KrylovEvolution(reg, layer.t_start:layer.step:layer.t_end, h)
      # question this does not exist so this has to implemented, any specific suggestions regarding this? I did found one funded by unitary fund https://github.com/emilianomfortes/krylovsolver
      # for (step, reg, _) in prob ## store the expectation of each operator for the given state in the output vector
      # question: how to obtain state in polynomial time since state tomography takes exponential measurements losing Q advantage
      # push!(readouts, chain(put(nsites, i => Z), put(nsites, j => Z))) ## what does this mean? Something like CZ?
      self.atoms @ np.exp(-1j * h * (self.t_end - self.t_start))

In [37]:
# need to figure out how to access this stuff
BloqadeEmulation()

TypeError: BloqadeEmulation.__init__() missing 1 required positional argument: 'task_data'

In [ ]:
#### How they do zero state

[emu] = (
    start.add_position([(0, 0), (0, 5), (0, 10)])
    .rydberg.rabi.amplitude.uniform.constant(15.0, 4.0)
    .detuning.uniform.constant(1.0, 4.0)
    .bloqade.python()
    .hamiltonian()
)

data = np.zeros(8)
data[0] = 1

emu.zero_state()

StateVector(data=array([1., 0., 0., 0., 0., 0., 0., 0.]), space=Space(space_type=<SpaceType.FullSpace: 'full_space'>, atom_type=TwoLevelAtomType(), program_register=Register(atom_type=TwoLevelAtomType(), sites=[(Decimal('0'), Decimal('0')), (Decimal('0'), Decimal('5')), (Decimal('0'), Decimal('10'))], blockade_radius=Decimal('0.0'), geometry=Geometry(sites=[(0.0, 0.0), (0.0, 5.0), (0.0, 10.0)], filling=[1, 1, 1], parallel_decoder=None), full_index_to_index={0: 0, 1: 1, 2: 2}), configurations=array([0, 1, 2, 3, 4, 5, 6, 7], dtype=uint32)))

In [ ]:
# have a certain square lattice or something. Don't know how much the shape matters.
atoms = Chain(2, lattice_spacing="distance") # probably generate_sites(ChainLattice(), dim_pca; scale = d); # put atoms in a chain with 10 micron spacing
nsites = dim_pca
Ω = 2*np.pi
t_start = 0.0
t_end = 4.0
step = 0.5
#using space class we have to establish zero state for the register
reg = zero_state(nsites)
#readouts seems non-trivial
# layer = DetuningLayer()

NameError: name 'zero_state' is not defined

## Defining a NN model

In [ ]:
# Define neural network model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(dim_pca, 10)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return x
    # Train classical model using PCA features
    model_reg = Net()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_reg.parameters(), lr=0.01)
    for epoch in range(1000):
        for x, y in train_loader:
            x = x.view(-1, 28*28)
            x_pca = pca.transform(x.numpy())
            x_pca = torch.tensor(x_pca, dtype=torch.float32)
            y = torch.tensor(y, dtype=torch.long)
            optimizer.zero_grad()
            output = model_reg(x_pca)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()
    # Train QRC model using quantum reservoir computing
    pre_layer = DetuningLayer(atoms, readouts, Ω, t_start, t_end, step)
    model_qrc = Net()
    for epoch in range(1000):
        for x, y in train_loader:
            x = x.view(-1, 28*28)
            x_pca = pca.transform(x.numpy())
